# Preprocessing in PyTorch

This notebook will show how to do the preprocessing step of a machine learning model using the `pytorch` library in Python.

In [169]:
import pandas as pd
import torch
import torch.nn as nn
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
import math

## Dataset

The dataset used is California Housing, and in this exercise we will be using sklearn to import it.

In [170]:
data = fetch_california_housing()

In [171]:
df = pd.DataFrame(data.data, columns=data.feature_names)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   MedInc      20640 non-null  float64
 1   HouseAge    20640 non-null  float64
 2   AveRooms    20640 non-null  float64
 3   AveBedrms   20640 non-null  float64
 4   Population  20640 non-null  float64
 5   AveOccup    20640 non-null  float64
 6   Latitude    20640 non-null  float64
 7   Longitude   20640 non-null  float64
dtypes: float64(8)
memory usage: 1.3 MB


In [172]:
# 80/10/10
x_train, x_temp, y_train, y_temp = train_test_split(df, data.target, test_size=0.2, random_state=42)
x_val, x_test, y_val, y_test = train_test_split(x_temp, y_temp, test_size=0.5, random_state=42)

In [173]:
print(f"Train size: {len(x_train)}")
print(f"Val size:   {len(x_val)}")
print(f"Test size:  {len(x_test)}")

Train size: 16512
Val size:   2064
Test size:  2064


In [174]:
x_train = torch.from_numpy(x_train.values).float()
x_val   = torch.from_numpy(x_val.values).float()
x_test  = torch.from_numpy(x_test.values).float()
y_train = torch.from_numpy(y_train).float().unsqueeze(1)
y_val   = torch.from_numpy(y_val).float().unsqueeze(1)
y_test  = torch.from_numpy(y_test).float().unsqueeze(1)

## EDA

For all intents and purposes, this housing dataset has enough, properly distributed data across the board. As we have done an exploratory analysis in previous notebooks, we will skip the analysis of the dataset.

However, we have previously made the following findings, which will be useful in our preprocessing step:
- There are 8 continuous features, all numbers `float64`
- There are no missing values, so we do not have to drop any columns or impute any values.
- `AveRooms`, `AveBedrms`, `Population`, `AveOccup` all have outliers.
- Some distributions are skewed left or right, but not in any meaningful way.
- `MedInc` had the most explainability in terms of the target variable.

## Preprocessing

For preprocessing, we will create a module using `nn.Module`.

We are careful not to introduce data leakage. Which is why we separated the __training__ and __test__ data before we do our preprocessing, to avoid leaking the __test__ data into our model. Having done this, we can avoid interacting with the test set completely until it is time for evaluation.

The instructions given for the preprocessing step of this model are the following:
- Avoid _data leakage_ (already done by splitting the sets)
- Exclude _outliers_, except in `Latitude` and `Longitude`
- Apply _standard scaling_, except in `Latitude` and `Longitude`
- Passthrough `Latitude` and `Longitude`

Since the first point has already been fulfilled, we can skip to the second. We will be creating a module for each "step" of the preprocessing stage, which will be introduced to the module as layers before any other.

The first layer is the **Outlier Remover**. It computes the IQR by obtaining the difference between the Q1 and Q3, then applying the formula for upper and lower bounds.

This is done in the constructor so it is only done once for the train dataset.

Every time this layer is stepped through, the data will be clamped between the computed bounds.

In [175]:
class OutlierRemover(nn.Module):
    def __init__(self, x: torch.Tensor):
        super().__init__()
        # let's avoid Latitude and Longitude first
        included = x[:, :6]

        # first quantile and third quantile (25% and 75%)
        # this formula was seen in the previous california housing notebook
        q1 = torch.quantile(included, 0.25, dim=0)
        q3 = torch.quantile(included, 0.75, dim=0)
        iqr = q3 - q1

        # we put the bounds in the buffer
        self.register_buffer('upper', q3 + 1.5 * iqr)
        self.register_buffer('lower', q1 - 1.5 * iqr)

    def forward(self, x) -> torch.Tensor:
        # this just clamps between the lower and upper bound computed above
        # we also avoid Latitude and Longitude here
        included = x[:, :6].clamp(self.lower, self.upper)

        # we concatenate the included features with the Latitude and Longitude
        return torch.cat([included, x[:, 6:]], dim=1)

The second layer is the **ScalingLayer**. It only applies the standard scaling formula when the layer is passed. It uses the values stored in the pytorch buffers. These values are computed, once again, the first time in the constructor.

In [176]:
class ScalingLayer(nn.Module):
    def __init__(self, x: torch.Tensor):
        super().__init__()
        # let's avoid Latitude and Longitude first
        included = x[:, :6] 

        # buffers so values are stored
        self.register_buffer('mean', included.mean(dim=0))
        self.register_buffer('std',  included.std(dim=0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # this is the formula for standard scaling: (x - mean) / std
        # let's avoid Latitude and Longitude first
        included = (x[:, :6] - self.mean) / self.std
        # we concatenate the included features with the Latitude and Longitude
        return torch.cat([included, x[:, 6:]], dim=1)

And finally, the last step of the preprocessing is to avoid Latitude and Longitude, as we want those values to be passed raw because they are coordinates, and their magnitude has no significant impact. 

To do this, we simply extracted the features we DO want to use `included = x[:, :6]` and only acted upon those, avoiding Latitude and Longitude. This is true for both the computing in the constructors, and the `forward` function which mutates the data.

Particularily, in `forward`, we have to concatenate the mutated data with the remainder (Latitude and Longitude themselves) because otherwise data would be missing. **NOTE: I USED AI FOR THIS CONCATENATION STEP, I DID NOT KNOW HOW TO CONCATENATE THE ARRAYS**

## Training

To train the model, we can just make a network using `nn.Sequential` and inputing our preprocessing layers (`OutlierRemover` and `ScalingLayer`) before any of the actual network layers.

First, we can make a reusable, sequentual pipeline that we can use in all of our networks before we compare performances. 

Note that we call `outlier_remover` on `x_train` and pass that to the scaling layer. This is because outliers would alter the *std* and *mean* of the Scaling Layer.

In [177]:
outlier_remover = OutlierRemover(x_train)
x_train_clamped = outlier_remover(x_train)

preprocessor = nn.Sequential(
    outlier_remover,
    ScalingLayer(x_train_clamped)
)

This is my training function from a different exercise. I will import it and omit explanation as the focus of this notebook is on the preprocessing step.

In [178]:
def evaluate(model, loss_fn, xval, yval):
    model.eval()
    with torch.no_grad():
        val_outputs = model(torch.from_numpy(xval).float())
        val_loss = loss_fn(val_outputs, torch.from_numpy(yval).float())
    return val_loss.item()

def train(model, optimizer, loss_fn, xtrain, ytrain, xval, yval, epochs=10, batch_size=32, delta=0.005, patience=3):
    val_loss_history = []

    # for each epoch
    for epoch in range(epochs):
        # we train the model
        model.train()

        # loop over mini batches
        for i in range(0, len(xtrain), batch_size):
            batch_x = torch.from_numpy(xtrain[i:i+batch_size]).float()
            batch_y = torch.from_numpy(ytrain[i:i+batch_size]).float()

            # compute outputs to measure the loss (forward pass)
            outputs = model(batch_x)
            loss = loss_fn(outputs, batch_y)

            # reset the GRADIENTS to zero
            optimizer.zero_grad()

            # compute the GRADIENTS (backward pass)
            loss.backward()

            # update the WEIGHTS, as they were modified by the backward pass
            # this is what persists across iterations and epochs
            optimizer.step()

        # then we evaluate the model on the validation set
        val_loss = evaluate(model, loss_fn, xval, yval)
        val_loss_history.append(val_loss)

        print(f'Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}, Val Loss: {val_loss:.4f}')

        # early stopping
        if len(val_loss_history) > patience:
            loss_n_patience = val_loss_history[-patience-1]
            loss_n = val_loss_history[-1]
            current_delta = abs(loss_n_patience - loss_n)
            if current_delta < delta and loss_n_patience >= loss_n:
                print(f"Early stopping at epoch {epoch+1}")
                break

We will use `MSELoss` as the loss function, as this is a regression problem.

In [179]:
loss_fn = nn.MSELoss()

### First Model

In [180]:
class FirstModel(nn.Module):
    def __init__(self, preprocessor: nn.Module):
        super().__init__()
        self.preprocessor = preprocessor
        self.network = nn.Sequential(
            nn.Linear(8, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.preprocessor(x)
        return self.network(x)

- Epochs: `100`
- Optim: `Adam`
- Learning Rate: `0.0001`
- Architecture:
    - Input: `8`
    - Hidden: `64`
    - Output: `1`

In [181]:
model1 = FirstModel(preprocessor)
optim1 = torch.optim.Adam(model1.parameters(), lr=0.0001)
epochs = 100
train(model1, optim1, loss_fn, x_train.numpy(), y_train.numpy(), x_val.numpy(), y_val.numpy(), epochs=epochs)

Epoch 1/100, Loss: 0.8104, Val Loss: 0.9218
Epoch 2/100, Loss: 0.6508, Val Loss: 0.7383
Epoch 3/100, Loss: 0.5531, Val Loss: 0.6385
Epoch 4/100, Loss: 0.4968, Val Loss: 0.5890
Epoch 5/100, Loss: 0.4676, Val Loss: 0.5605
Epoch 6/100, Loss: 0.4496, Val Loss: 0.5420
Epoch 7/100, Loss: 0.4405, Val Loss: 0.5287
Epoch 8/100, Loss: 0.4343, Val Loss: 0.5189
Epoch 9/100, Loss: 0.4306, Val Loss: 0.5118
Epoch 10/100, Loss: 0.4295, Val Loss: 0.5066
Epoch 11/100, Loss: 0.4294, Val Loss: 0.5025
Epoch 12/100, Loss: 0.4296, Val Loss: 0.4993
Epoch 13/100, Loss: 0.4301, Val Loss: 0.4970
Epoch 14/100, Loss: 0.4308, Val Loss: 0.4950
Epoch 15/100, Loss: 0.4314, Val Loss: 0.4935
Epoch 16/100, Loss: 0.4319, Val Loss: 0.4921
Early stopping at epoch 16


### Second Model

In [182]:
class SecondModel(nn.Module):
    def __init__(self, preprocessor: nn.Module):
        super().__init__()
        self.preprocessor = preprocessor
        self.network = nn.Sequential(
            nn.Linear(8, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.preprocessor(x)
        return self.network(x)

- Epochs: `100`
- Optim: `Adam`
- Learning Rate: `0.001`
- Architecture:
    - Input: `8`
    - Hidden 1: `128`
    - Dropout: `0.2`
    - Hidden 2: `64`
    - Output: `1`

In [183]:
model2 = SecondModel(preprocessor)
optim2 = torch.optim.Adam(model2.parameters(), lr=0.001)
train(model2, optim2, loss_fn, x_train.numpy(), y_train.numpy(), x_val.numpy(), y_val.numpy(), epochs=100)

Epoch 1/100, Loss: 0.6301, Val Loss: 0.7341
Epoch 2/100, Loss: 0.4826, Val Loss: 0.5298
Epoch 3/100, Loss: 0.4326, Val Loss: 0.4780
Epoch 4/100, Loss: 0.4655, Val Loss: 0.4691
Epoch 5/100, Loss: 0.3466, Val Loss: 0.4631
Epoch 6/100, Loss: 0.3650, Val Loss: 0.4611
Epoch 7/100, Loss: 0.4304, Val Loss: 0.4608
Epoch 8/100, Loss: 0.4452, Val Loss: 0.4710
Epoch 9/100, Loss: 0.4091, Val Loss: 0.4514
Epoch 10/100, Loss: 0.3909, Val Loss: 0.4480
Epoch 11/100, Loss: 0.5025, Val Loss: 0.4442
Epoch 12/100, Loss: 0.4097, Val Loss: 0.4518
Epoch 13/100, Loss: 0.4286, Val Loss: 0.4452
Early stopping at epoch 13


### Third Model

In [184]:
class ThirdModel(nn.Module):
    def __init__(self, preprocessor: nn.Module):
        super().__init__()
        self.preprocessor = preprocessor
        self.network = nn.Sequential(
            nn.Linear(8, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.preprocessor(x)
        return self.network(x)

- Epochs: `100`
- Optim: `SGD`
- Learning Rate: `0.01`
- Architecture:
    - Input: `8`
    - Hidden: `32`
    - Dropout: `0.3`
    - Output: `1`

In [185]:
model3 = ThirdModel(preprocessor)
optim3 = torch.optim.SGD(model3.parameters(), lr=0.01)
train(model3, optim3, loss_fn, x_train.numpy(), y_train.numpy(), x_val.numpy(), y_val.numpy(), epochs=100)

Epoch 1/100, Loss: 1.1065, Val Loss: 1.3187
Epoch 2/100, Loss: 1.1060, Val Loss: 1.3189
Epoch 3/100, Loss: 1.1060, Val Loss: 1.3189
Epoch 4/100, Loss: 1.1060, Val Loss: 1.3189
Epoch 5/100, Loss: 1.1060, Val Loss: 1.3189
Early stopping at epoch 5


## Model Comparison

To compare the models we can use RMSE to find the best one. 

In [186]:
def eval(model, loss_fn, x, y):
    model.eval() # exit train
    with torch.no_grad():
        preds = model(x)
        mse = loss_fn(preds, y).item()
    return mse, math.sqrt(mse) # simple formula

This evaluation process is taken from my previous `3.5 Neural Networks` exercise.

In [187]:
loss_fn = nn.MSELoss()

mse1, rmse1 = eval(model1, loss_fn, x_val, y_val)
mse2, rmse2 = eval(model2, loss_fn, x_val, y_val)
mse3, rmse3 = eval(model3, loss_fn, x_val, y_val)

print("Validation Set Results:")
print(f"Model 1 - MSE: {mse1:.4f}, RMSE: {rmse1:.4f}")
print(f"Model 2 - MSE: {mse2:.4f}, RMSE: {rmse2:.4f}")
print(f"Model 3 - MSE: {mse3:.4f}, RMSE: {rmse3:.4f}")

Validation Set Results:
Model 1 - MSE: 0.4921, RMSE: 0.7015
Model 2 - MSE: 0.4452, RMSE: 0.6673
Model 3 - MSE: 1.3189, RMSE: 1.1484


In [188]:
models = [model1, model2, model3]
val_mse = [mse1, mse2, mse3]
val_rmse = [rmse1, rmse2, rmse3]
best_idx = val_mse.index(min(val_mse))
best_model = models[best_idx]

print(f"Best model: Model {best_idx + 1}")
print(f"Val MSE:  {val_mse[best_idx]:.4f}")
print(f"Val RMSE: {val_rmse[best_idx]:.4f}")

test_mse, test_rmse = eval(best_model, loss_fn, x_test, y_test)
print(f"\nTest results:")
print(f"Test MSE:  {test_mse:.4f}")
print(f"Test RMSE: {test_rmse:.4f}")

Best model: Model 2
Val MSE:  0.4452
Val RMSE: 0.6673

Test results:
Test MSE:  0.4377
Test RMSE: 0.6616


Finally, the best out of the three networks was **Model 2**